Generate Candlestick Images

In [6]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import ViTImageProcessor, ViTModel
from PIL import Image
import mplfinance as mpf
import os
from tqdm import tqdm
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt

df = pd.read_csv("nvda_prices_with_sentiment.csv", 
                 index_col="Date", parse_dates=True, dayfirst=True)

print("Loaded shape:", df.shape)
print(df.tail(5)[['Close', 'finbert_sentiment']])

image_dir = "candlestick_images"
os.makedirs(image_dir, exist_ok=True)

def save_candlestick(row_index, df):
    row_date = df.index[row_index]
    
    # Safety: make sure row_date is datetime
    if isinstance(row_date, str):
        row_date = pd.to_datetime(row_date, dayfirst=True, errors='coerce')
    elif not isinstance(row_date, (pd.Timestamp, datetime.datetime)):
        print(f"Skipping row {row_index}: row_date is not datetime or str")
        return None
    
    window = df.iloc[row_index-59:row_index+1].copy()
    
    if len(window) < 60:
        return None
    
    # Fix window index (local copy)
    window.index = pd.to_datetime(window.index, errors='coerce', dayfirst=True)
    window = window[window.index.notna()]
    
    fig, ax = plt.subplots(figsize=(6, 4))
    mpf.plot(window, type='candle', style='yahoo', ax=ax, volume=False, show_nontrading=False)
    plt.close(fig)
    
    # Safe strftime
    date_str = row_date.strftime('%Y%m%d') if hasattr(row_date, 'strftime') else "unknown_date"
    save_path = os.path.join(image_dir, f"nvda_{date_str}.png")
    fig.savefig(save_path, dpi=100, bbox_inches='tight')
    
    return save_path


# Calculate max possible starting index
max_start = len(df) - 60

# Limit num_images to available windows
num_images = min(num_images, max_start)

print(f"Generating images for {num_images} windows (max possible: {max_start})")

image_paths = []
for i in tqdm(range(max_start - num_images, max_start)):
    path = save_candlestick(i, df)
    if path:
        image_paths.append((df.index[i], path))

print(f"Generated {len(image_paths)} images")

Loaded shape: (2566, 7)
                 Close  finbert_sentiment
Date                                     
2026-03-11  186.029999           0.000000
2026-03-12  183.139999           0.000000
2026-03-13  180.250000          -0.736447
2026-03-16  183.220001           0.000000
2026-03-17  181.929993          -0.109047
Generating images for 1000 windows (max possible: 2506)


  0%|          | 0/1000 [00:00<?, ?it/s]

C:\Users\sam4s\AppData\Local\Temp\ipykernel_20384\68478754.py:28: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  row_date = pd.to_datetime(row_date, dayfirst=True, errors='coerce')
C:\Users\sam4s\AppData\Local\Temp\ipykernel_20384\68478754.py:39: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  window.index = pd.to_datetime(window.index, errors='coerce', dayfirst=True)
C:\Users\sam4s\AppData\Local\Temp\ipykernel_20384\68478754.py:28: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  row_date = pd.to_datetime(row_date, dayfirst=True, errors='coerce')
  0%|          | 2/1000 [00:00<01:24, 11.79it/s]C:\Users\sam4s\AppData\Local\Temp\ipykernel_20384\68478754.py:28: UserWarning: Parsing dates in

Generated 1000 images


Load ViT model & processor

In [7]:
from transformers import ViTImageProcessor, ViTModel
import torch
from PIL import Image
from tqdm import tqdm

processor = ViTImageProcessor.from_pretrained('google/vit-base-patch16-224')
vit_model = ViTModel.from_pretrained('google/vit-base-patch16-224')
vit_model.eval()  # inference mode
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
vit_model.to(device)

print("ViT model loaded on:", device)

C:\Users\sam4s\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sam4s\.cache\huggingface\hub\models--google--vit-base-patch16-224. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100

ViT model loaded on: cpu


 Extract ViT features from all generated images

In [8]:
vit_features = {}

for date, path in tqdm(image_paths):
    try:
        image = Image.open(path).convert("RGB")
        inputs = processor(images=image, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = vit_model(**inputs)
        
        # Take [CLS] token (first token, 768-dim feature vector)
        cls_feature = outputs.last_hidden_state[:, 0, :].cpu().numpy().flatten()
        vit_features[date] = cls_feature
        
    except Exception as e:
        print(f"Error on {date} ({path}): {e}")

print(f"\nExtracted ViT features for {len(vit_features)} images")

100%|██████████| 1000/1000 [04:22<00:00,  3.80it/s]


Extracted ViT features for 1000 images


Merge ViT features into df

In [9]:
vit_df = pd.DataFrame.from_dict(vit_features, orient='index')
vit_df.columns = [f"vit_dim_{i}" for i in range(vit_df.shape[1])]

df = df.join(vit_df, how='inner')

print("Merged shape with ViT features:", df.shape)
print("New ViT columns:", list(df.columns[-5:]))  # last 5 ViT columns

df.to_csv("nvda_prices_sentiment_vit.csv")
print("Full multimodal data saved.")

Merged shape with ViT features: (1000, 775)
New ViT columns: ['vit_dim_763', 'vit_dim_764', 'vit_dim_765', 'vit_dim_766', 'vit_dim_767']
Full multimodal data saved.


Load ViT processor & model

In [10]:
from transformers import ViTImageProcessor, ViTModel
from tqdm import tqdm
import torch
from PIL import Image

processor = ViTImageProcessor.from_pretrained('google/vit-base-patch16-224')
vit_model = ViTModel.from_pretrained('google/vit-base-patch16-224')
vit_model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
vit_model.to(device)

print("ViT loaded on:", device)

Loading weights: 100%|██████████| 198/198 [00:00<00:00, 6200.01it/s]
ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.weight   | UNEXPECTED | 
classifier.bias     | UNEXPECTED | 
pooler.dense.bias   | MISSING    | 
pooler.dense.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


ViT loaded on: cpu


Function to extract features from one image

In [11]:
def extract_vit_features(image_path):
    try:
        image = Image.open(image_path).convert("RGB")
        inputs = processor(images=image, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = vit_model(**inputs)
        
        # CLS token (768-dim feature)
        cls_feat = outputs.last_hidden_state[:, 0, :].cpu().numpy().flatten()
        return cls_feat
    except Exception as e:
        print(f"Error on {image_path}: {e}")
        return None

Extract features for all images

In [12]:
vit_features = {}

for date, path in tqdm(image_paths):
    feat = extract_vit_features(path)
    if feat is not None:
        vit_features[date] = feat

print(f"Extracted features for {len(vit_features)} images")

100%|██████████| 1000/1000 [04:40<00:00,  3.57it/s]

Extracted features for 1000 images


Merge ViT features into main DataFrame

In [13]:
vit_df = pd.DataFrame.from_dict(vit_features, orient='index')
vit_df.columns = [f"vit_{i}" for i in range(vit_df.shape[1])]

df = df.join(vit_df, how='inner')

print("Final shape with ViT:", df.shape)
df.to_csv("nvda_full_multimodal.csv")
print("Saved full data with ViT features.")

Final shape with ViT: (1000, 1543)
Saved full data with ViT features.


In [16]:
df = pd.read_csv("nvda_prices_sentiment_vit.csv", index_col=[0], parse_dates=True, dayfirst=True)
print("Full multimodal shape:", df.shape)
print("Last 5 rows preview:")
print(df.tail(5)[['Close', 'finbert_sentiment'] + [col for col in df.columns if col.startswith('vit_')][:3]])

Full multimodal shape: (1000, 775)
Last 5 rows preview:
                 Close  finbert_sentiment  vit_dim_0  vit_dim_1  vit_dim_2
2025-12-11  180.920197                0.0   0.195352  -0.440696   0.217055
2025-12-12  175.010529                0.0   0.159925  -0.282702   0.469230
2025-12-15  176.280457                0.0   0.247811  -0.377223   0.789828
2025-12-16  177.710388                0.0   0.330134  -0.345409   0.637049
2025-12-17  170.930756                0.0   0.285559  -0.261926   0.571585


C:\Users\sam4s\AppData\Local\Temp\ipykernel_20384\1972400827.py:1: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df = pd.read_csv("nvda_prices_sentiment_vit.csv", index_col=[0], parse_dates=True, dayfirst=True)
